<h1 align="center"> Read Multiple Data Sets in a Single DataFrame </h1>

### Import Libraries

In [ ]:
import polars as pl
import numpy as np
from matplotlib import pyplot as plt
from IPython.display import display, HTML

%matplotlib inline
display(HTML("<style>.container { width:100% !important; }</style>"))

### Reading Submission Data Sets for All Quarters of 2022 from [Financial Statement Data Sets](https://www.sec.gov/dera/data/financial-statement-data-sets)

In [ ]:
filingsQ1 = pl.read_csv('data/2022q1/sub.txt', separator='\t', infer_schema_length=10000)
filingsQ2 = pl.read_csv('data/2022q2/sub.txt', separator='\t', infer_schema_length=10000)
filingsQ3 = pl.read_csv('data/2022q3/sub.txt', separator='\t', infer_schema_length=10000)
filingsQ4 = pl.read_csv('data/2022q4/sub.txt', separator='\t', infer_schema_length=10000)
print('Sum of Filings:', len(filingsQ1) + len(filingsQ2) + len(filingsQ3) + len(filingsQ4))

### Using pl.concat to Create a Single Filings DataFrame for All 4 Quarters

In [ ]:
filings = pl.concat([filingsQ1, filingsQ2, filingsQ3, filingsQ4])
print('Total num of Filings in filings DataFrame:', len(filings))

### Look at the Contents of the First Row of filings DataFrame

In [ ]:
filings[0:1]

### Construct a New Column Month using Filed Date (YYYYMMDD // 100 → YYYYMM)

In [ ]:
filings = filings.with_columns(
    (pl.col('filed') // 100).alias('Month')
)

### Look at the First Row — Notice the New Column Month in YYYYMM Format

In [ ]:
filings.head(1)

### Group Filings By Month Column and Compute Group Sizes

In [ ]:
filingsGroupedByMonth = (
    filings
    .group_by('Month')
    .agg(pl.len().alias('Num. of Filings'))
    .sort('Month')
)
filingsGroupedByMonth

### Generate Descriptive Statistics of filingsGroupedByMonth DataFrame

In [ ]:
filingsGroupedByMonth.describe()

### Visualize Monthly Statistics using Matplotlib Library

In [ ]:
filingsGroupedByMonth.to_pandas().plot(x='Month', y='Num. of Filings', kind='bar', figsize=(14,6))
plt.title('Monthly Structured Data Filing Statistics for 2022', fontsize=20)
plt.xlabel('Month', fontsize=15)
plt.ylabel('Number of Filings', fontsize=15)
plt.yticks(np.arange(0, 6001, step=500))
plt.xticks(rotation=45)
plt.savefig('images/MonthlyStats2022.jpg')
plt.show()